In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pybromo as pbm
from pybromo.diffusion import Particle   # classe non esposta in pybromo.__init__
import os, time, tables
import tomli
# directory to save HDF5
folder = './Pybromo_simulations'
os.makedirs(folder, exist_ok=True)
# import simulation parameters
params = tomli.load(open("parameters.toml", 'rb'))

## System setup: **BOX** & **LASER**

In [ ]:
# grant reproducibility by defining initial random state
rs = np.random.RandomState(seed=1)
# generate volume box
box = pbm.Box(
    x1 = params['box']['x1'], x2 = params['box']['x2'],
    y1 = params['box']['y1'], y2 = params['box']['y2'],
    z1 = params['box']['z1'], z2 = params['box']['z2']
)

# PSF: defined DIRECTLY as the EFFECTIVE observation volume
w0_eff = params['PSF']['w0_eff']   # effective 1/e^2 radius (xy)
z0_eff = params['PSF']['z0_eff']   # effective 1/e^2 radius (z)

sx = w0_eff / np.sqrt(2)         # sigma of the stored field Gaussian = effective radius
sy = w0_eff / np.sqrt(2)
sz = z0_eff / np.sqrt(2)

# calculate excitation volumes (confocal and box)
vol_psf = np.pi**1.5 * (w0_eff**2) * z0_eff   # ~ 3.4e-19 m^3
vol_box = (box.x2 - box.x1) * (box.y2 - box.y1) * (box.z2 - box.z1)

# creation of laser PSF
psf = pbm.GaussianPSF(
    xc=0, yc=0, zc=0,  # Box centering
    sx=sx, sy=sy, sz=sz
)

print(f"Box: {box.x2-box.x1:.0e} x {box.y2-box.y1:.0e} x {box.z2-box.z1:.0e} m^3")
print(f"Effective confocal PSF: w0_eff = {w0_eff*1e6:.3f} um, z0_eff = {z0_eff*1e6:.3f} um (kappa = {z0_eff/w0_eff:.1f})")
print(f"Confocal volume: vol_psf = {vol_psf:.3e} m^3")

# I. **Free EtBr** (100 molecules in the Box)

In [ ]:
# diffusion specs
EtBr_Du = params['particles']['D_etbr_free']          # µm²/s
EtBr_D = EtBr_Du*(1e-6)**2   # m²/s
# number of molecules in the box
n_particles = params['particles']['n_etbr_free']
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_check = n_particles * vol_psf / vol_box
print(f"N_box = {n_particles} molecules -> n_avg in confocal volume = {n_avg_check:.3f}")

In [ ]:
# generate particles in the simulation
P = pbm.Particles(
    num_particles = n_particles, D = EtBr_D, box = box, rs = rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, t_max=t_max, particles=P, box=box, psf=psf
)
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (6 DNA 'blobs' in the Box)

In [ ]:
# Population 1 (free EtBr) specs
N_EtBr_free = params['particles']['n_etbr_free']
D_free = params['particles']['D_etbr_free'] * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
# average number of 'blobs' of DNA (with bounded EtBr) in confocal volume
N_blobs = params['particles']['n_etbr_dna_006']
D_bound = params['particles']['D_etbr_dna'] * (1e-6)**2
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_free_check = N_EtBr_free * vol_psf / vol_box
n_avg_bound_check = N_blobs * vol_psf / vol_box
print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- N_box = {N_EtBr_free} -> n_avg in confocal vol = {n_avg_free_check:.3f}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- N_box = {N_blobs} -> n_avg in confocal vol = {n_avg_bound_check:.3f}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (8 DNA 'blobs' in the Box)

In [ ]:
# Population 1 (free EtBr) specs
N_EtBr_free = params['particles']['n_etbr_free']
D_free = params['particles']['D_etbr_free'] * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
# average number of 'blobs' of DNA (with bounded EtBr) in confocal volume
N_blobs = params['particles']['n_etbr_dna_008']
D_bound = params['particles']['D_etbr_dna'] * (1e-6)**2
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_free_check = N_EtBr_free * vol_psf / vol_box
n_avg_bound_check = N_blobs * vol_psf / vol_box
print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- N_box = {N_EtBr_free} -> n_avg in confocal vol = {n_avg_free_check:.3f}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- N_box = {N_blobs} -> n_avg in confocal vol = {n_avg_bound_check:.3f}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (10 DNA 'blobs' in the Box)

In [ ]:
# Population 1 (free EtBr) specs
N_EtBr_free = params['particles']['n_etbr_free']
D_free = params['particles']['D_etbr_free'] * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
N_blobs = params['particles']['n_etbr_dna_010']
D_bound = params['particles']['D_etbr_dna'] * (1e-6)**2
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_free_check = N_EtBr_free * vol_psf / vol_box
n_avg_bound_check = N_blobs * vol_psf / vol_box
print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- N_box = {N_EtBr_free} -> n_avg in confocal vol = {n_avg_free_check:.3f}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- N_box = {N_blobs} -> n_avg in confocal vol = {n_avg_bound_check:.3f}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (12 DNA 'blobs' in the Box)

In [ ]:
# Population 1 (free EtBr) specs
N_EtBr_free = params['particles']['n_etbr_free']
D_free = params['particles']['D_etbr_free'] * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
N_blobs = params['particles']['n_etbr_dna_012']
D_bound = params['particles']['D_etbr_dna'] * (1e-6)**2
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_free_check = N_EtBr_free * vol_psf / vol_box
n_avg_bound_check = N_blobs * vol_psf / vol_box
print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- N_box = {N_EtBr_free} -> n_avg in confocal vol = {n_avg_free_check:.3f}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- N_box = {N_blobs} -> n_avg in confocal vol = {n_avg_bound_check:.3f}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()

# II. **Bound EtBr-DNA** run (15 DNA 'blobs' in the Box)

In [ ]:
# Population 1 (free EtBr) specs
N_EtBr_free = params['particles']['n_etbr_free']
D_free = params['particles']['D_etbr_free'] * (1e-6)**2   # m²/s

# Population 2 (bounded EtBr-DNA) specs
N_blobs = params['particles']['n_etbr_dna_015']
D_bound = params['particles']['D_etbr_dna'] * (1e-6)**2
# time simulation parameters
t_step = params['simulation']['t_step']   # time step (s), t
t_max = params['simulation']['t_max']       # total simulation (s)

n_avg_free_check = N_EtBr_free * vol_psf / vol_box
n_avg_bound_check = N_blobs * vol_psf / vol_box
print(f'EtBr-free specs:\n- Diffusion: {D_free} m^2/s\n- N_box = {N_EtBr_free} -> n_avg in confocal vol = {n_avg_free_check:.3f}')
print(f'EtBr-DNA  specs:\n- Diffusion: {D_bound} m^2/s\n- N_box = {N_blobs} -> n_avg in confocal vol = {n_avg_bound_check:.3f}')

In [ ]:
# generate particles in the simulation
P = pbm.Particles.from_specs(
    num_particles=(N_EtBr_free, N_blobs),
    D=(D_free, D_bound),
    box=box, rs=rs
)
# prepare simulation timing before running it
S = pbm.ParticlesSimulation(
    t_step=t_step, 
    t_max=t_max, 
    particles=P, 
    box=box, 
    psf=psf
    )
# display expected size (general memory information)
print(S.print_sizes())

In [ ]:
# execute simulation 
S.simulate_diffusion(
    save_pos=False,           # <-- True if we want to record and save positions
    total_emission=False,    # <-- False if we want to generate timestamps
    rs=rs,
    chunksize=2**19,
    chunkslice='times',
    path = folder
)
print('Shape emission:', S.emission.shape)  #expected: (n_particles, n_timepoints)

In [ ]:
# once simulation is done, it is better to close the HDF5 file to prevent issues in the workflow
# (the file will be opened again in the notebook 2 to generate photon timestamps)
S.store.close()